# Change Preventers — frenan el cambio

Modificar el sistema se vuelve pesado: un cambio exige múltiples cambios en cascada.

## Serie: Refactorización y Code Smells

Este contenido está dividido en 6 notebooks — uno por categoría de code
smell (clasificación de refactoring.guru) más un cierre de ejercicios:

1. `01_bloaters.ipynb` — Bloaters
2. `02_object_orientation_abusers.ipynb` — Object-Orientation Abusers
3. **`03_change_preventers.ipynb`** — Change Preventers
4. `04_dispensables.ipynb` — Dispensables
5. `05_couplers.ipynb` — Couplers
6. `06_ejercicios_autoevaluacion.ipynb` — Ejercicios y autoevaluación


### 1. Shotgun Surgery (cirugía de escopeta)

**Definición:** Un solo cambio de regla de negocio obliga a modificar pequeñas partes en muchas clases distintas del sistema.

**Síntoma:** La misma constante o regla de negocio está copiada en múltiples clases no relacionadas.

**Técnica de refactor:** Move Method / Move Field (centralizar la regla)

#### Con el smell

In [ ]:
class Factura:
    def total_con_iva(self, subtotal):
        return subtotal * 1.19  # 19% quemado

class Reporte:
    def calcular_iva(self, monto):
        return monto * 0.19  # duplicado

class Carrito:
    def impuesto(self, monto):
        return monto * 0.19  # duplicado otra vez

print(Factura().total_con_iva(100))
print(Reporte().calcular_iva(100))
print(Carrito().impuesto(100))

#### Refactorizado

In [ ]:
class ConfiguracionImpuestos:
    TASA_IVA = 0.19

    @classmethod
    def iva(cls, monto):
        return monto * cls.TASA_IVA

class Factura:
    def total_con_iva(self, subtotal):
        return subtotal + ConfiguracionImpuestos.iva(subtotal)

class Reporte:
    def calcular_iva(self, monto):
        return ConfiguracionImpuestos.iva(monto)

class Carrito:
    def impuesto(self, monto):
        return ConfiguracionImpuestos.iva(monto)

print(Factura().total_con_iva(100))
print(Reporte().calcular_iva(100))
print(Carrito().impuesto(100))

**Explicación:** `ConfiguracionImpuestos` concentra la regla de negocio (la tasa de IVA). Cambiar el IVA de 19% a 20% ahora significa editar una sola línea, en un solo lugar, en vez de perseguir cada copia por el código.

### 2. Divergent Change (cambio divergente)

**Definición:** Una misma clase cambia de formas totalmente distintas según cuál sea la razón del cambio.

**Síntoma:** Modificar una regla de nómina y modificar el motor de persistencia terminan tocando la misma clase — dos razones de cambio mezcladas.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

    def calcular_pago_mensual(self):
        return self.salario / 12

    def guardar_en_base_datos(self):
        # cambia si cambia el motor de BD, no la nómina
        print(f"INSERT INTO empleados VALUES ('{self.nombre}')")

empleado = Empleado("Ana", 3_600_000)
print(empleado.calcular_pago_mensual())
empleado.guardar_en_base_datos()

#### Refactorizado

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

class CalculadoraSalario:
    def pago_mensual(self, empleado):
        return empleado.salario / 12

class EmpleadoRepositorio:
    def guardar(self, empleado):
        print(f"INSERT INTO empleados VALUES ('{empleado.nombre}')")

empleado = Empleado("Ana", 3_600_000)
print(CalculadoraSalario().pago_mensual(empleado))
EmpleadoRepositorio().guardar(empleado)

**Explicación:** `Extract Class` separa las dos razones de cambio: `CalculadoraSalario` cambia solo si cambian las reglas de nómina, `EmpleadoRepositorio` cambia solo si cambia el motor de persistencia. `Empleado` ya no cambia por ninguna de las dos razones.

### 3. Parallel Inheritance Hierarchies (jerarquías de herencia paralelas)

**Definición:** Cada vez que se crea una subclase en una jerarquía, hay que crear también una subclase correspondiente en otra jerarquía distinta, solo para que las dos "vayan de la mano".

**Síntoma:** Los nombres de las clases en dos jerarquías se parecen sospechosamente (`Empleado`/`Gerente`/`Vendedor` y `ReporteEmpleado`/`ReporteGerente`/`ReporteVendedor`) y agregar un tipo nuevo siempre implica tocar ambas.

**Técnica de refactor:** Move Method / Move Field (fusionar las jerarquías)

#### Con el smell

In [ ]:
class Empleado:
    pass

class Gerente(Empleado):
    pass

class Vendedor(Empleado):
    pass

class ReporteEmpleado:
    def generar(self, empleado):
        return f"Reporte genérico de {type(empleado).__name__}"

class ReporteGerente(ReporteEmpleado):
    def generar(self, empleado):
        return f"Reporte gerencial de {type(empleado).__name__}"

class ReporteVendedor(ReporteEmpleado):
    def generar(self, empleado):
        return f"Reporte de ventas de {type(empleado).__name__}"

print(ReporteGerente().generar(Gerente()))

#### Refactorizado

In [ ]:
class Empleado:
    def generar_reporte(self):
        return f"Reporte genérico de {type(self).__name__}"

class Gerente(Empleado):
    def generar_reporte(self):
        return f"Reporte gerencial de {type(self).__name__}"

class Vendedor(Empleado):
    def generar_reporte(self):
        return f"Reporte de ventas de {type(self).__name__}"

print(Gerente().generar_reporte())

**Explicación:** `Move Method`/`Move Field` trasladan el comportamiento de la jerarquía `ReporteX` a la jerarquía `Empleado` original, fusionándolas en una sola. Agregar un nuevo tipo de empleado ahora solo exige tocar una jerarquía, no dos.